In [1]:
import pandas as pd
import pickle
import numpy as np
import time
import random
import igraph as ig
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
from operator import itemgetter
from sklearn.preprocessing import MinMaxScaler
from SOINN import SOINN

In [2]:
def train_phase(model, data, labels):
    # for plotting
    xs = []
    n_nodes = []
    n_edges = []
    n_del_nodes = []
    n_del_edges = []

    start_time = time.time()

    for index, row in data.iterrows():
        model.input_signal(x=row.values, y=labels[index], learning=True)

        # print completed percentage
        percent_completed = round((index / len(data))*100, 2)
        if index % 100 == 0 or index == len(data):
            clear_output(wait=True)
            print(f'Processing input {index}: {percent_completed}% completed')
            # for plotting
            n_nodes.append(model.network.vcount())
            n_edges.append(model.network.ecount())
            n_del_nodes.append(model.n_del_nodes)
            n_del_edges.append(model.n_del_edges)
            xs.append(index)

    finish_time = round(time.time() - start_time)
    mins = round(finish_time / 60)
    secs = finish_time % 60
    clear_output(wait=True)
    print(f'Training time: {mins} min {secs} sec')
    print(f'Inputs processed: {index}')

    print(f'Number of nodes: {model.network.vcount()}')
    print(f'Number of edges: {model.network.ecount()}')

    return xs, n_nodes, n_edges, n_del_nodes, n_del_edges

In [3]:
def live_phase(model, data, labels, plots=False):
    predicted = []
    xs = []
    accuracies = []

    positives = 0
    negatives = 0
    true_positives = 0
    true_negatives = 0

    for index, row in data.iterrows():
        x = row.values
        y = labels[index]
        yp = model.input_signal(x=x, learning=False)
        # append predicted label for later visualization
        predicted.append(yp)

        # feedback loop
        if y != yp:
            model.input_signal(x=x, y=y, learning=True)

        # error types
        if yp == 'attack' and y == 'attack':
            true_positives += 1
        if yp == 'normal' and y == 'normal':
            true_negatives += 1
        if y == 'normal':
            positives += 1
        if y == 'attack':
            negatives += 1
        
        # print completed percentage
        percent_completed = round((index / len(data))*100, 2)
        if index % 100 == 0 or index == len(data):
            clear_output(wait=True)
            print(f'Predictions completed: {percent_completed}%')
            if index != 0:
                xs.append(index)
                a = round((true_positives + true_negatives) / (positives + negatives) * 100, 2)
                accuracies.append(a)

    clear_output(wait=True)
    print(f'Inputs processed: {index}')
    a = round((true_positives + true_negatives) / (positives + negatives), 2)
    print(f'Overall accuracy: {a}%')

    # creating a frequency dict for visualizing the attacks with the most mistakes
    errors_freq = dict()
    for i in range(len(labels)):
        if labels[i] != predicted[i]:
            errors_freq[f'{labels[i]}-{predicted[i]}'] = 0
    for i in range(len(labels)):
        if labels[i] != predicted[i]:
            errors_freq[f'{labels[i]}-{predicted[i]}'] += 1
    # sort frequency dictionary
    errors_freq = dict(sorted(errors_freq.items(), key=itemgetter(1), reverse=True))
    print('\n\n')
    print(errors_freq)

    # plotting
    if plots:
        print('\n\n')
        plt.plot(xs, accuracies, label='accuracy')
        plt.title('Accuracy rate during live phase')
        plt.xlabel('inputs')
        plt.ylabel('percentage')
        plt.legend()
        plt.show()

In [4]:
# also biflow data is available
normal = pd.read_csv('./MQTT-IoT-IDS2020/uniflow_normal.csv')
bruteforce = pd.read_csv('./MQTT-IoT-IDS2020/uniflow_mqtt_bruteforce.csv')
scan_a = pd.read_csv('./MQTT-IoT-IDS2020/uniflow_scan_A.csv')
scan_su = pd.read_csv('./MQTT-IoT-IDS2020/uniflow_scan_sU.csv')
sparta = pd.read_csv('./MQTT-IoT-IDS2020/uniflow_sparta.csv')

# need to delete before append
del normal['ip_src']
del normal['ip_dst']
del bruteforce['ip_src']
del bruteforce['ip_dst']
del scan_a['ip_src']
del scan_a['ip_dst']
del scan_su['ip_src']
del scan_su['ip_dst']
del sparta['ip_src']
del sparta['ip_dst']

print(f'Normal connections: {len(normal)}')
print(f'MQTT bruteforce connections: {len(bruteforce)}')
print(f'Scan A connections: {len(scan_a)}')
print(f'Scan sU connections: {len(scan_su)}')
print(f'Sparta connections: {len(sparta)}')

# adding class label and removing boolean column for attacks
normal['class'] = 'normal'
del normal['is_attack']
bruteforce['class'] = 'attack'
del bruteforce['is_attack']
scan_a['class'] = 'attack'
del scan_a['is_attack']
scan_su['class'] = 'attack'
del scan_su['is_attack']
sparta['class'] = 'attack'
del sparta['is_attack']

# combining dataframes into one big dataframe
df = normal.append([bruteforce, scan_a, scan_su, sparta])
df = df.fillna(0)

# randomly sorting the dataframe
df = df.sample(frac=1).reset_index(drop=True)

# dropping src and dst ports
del df['prt_src']
del df['prt_dst']
del df['proto']

# one-hot-encoding for protocol and ports columns
#df = pd.get_dummies(df, columns=['proto'])

# divide into train and test
train = df[:150000]
train.reset_index(inplace=True)
del train['index']
y_train = train['class'].values
del train['class']

test = df[150000:]
test.reset_index(inplace=True)
del test['index']
y_test = test['class'].values
del test['class']

Normal connections: 171836
MQTT bruteforce connections: 33079
Scan A connections: 51358
Scan sU connections: 56845
Sparta connections: 182407


In [5]:
rand_int = random.randint(1, len(train) - 1)
x1 = train.iloc[rand_int].values
rand_int = random.randint(1, len(train) - 1)
x2 = train.iloc[rand_int].values
rand_int = random.randint(1, len(train) - 1)
x3 = train.iloc[rand_int].values

s = SOINN.SOINN(x1, x2, x3, max_edge_age=10000, iter_lambda=100)

xs, n_nodes, n_edges, n_del_nodes, n_del_edges = train_phase(model=s, data=train, labels=y_train)

Training time: 7 min 36 sec
Inputs processed: 149999
Number of nodes: 401
Number of edges: 378


In [ ]:
live_phase(model=s, data=test, labels=y_test, plots=True)

In [ ]:
i = 0
color_dict = dict()
classes = np.unique(y_train)
palette = ig.ClusterColoringPalette(len(classes))
for c in classes:
    color_dict[c] = palette[i]
    i += 1

ig.plot(s.network, vertex_size=5, vertex_color=[color_dict[c] for c in s.network.vs['c']])